## Landing to bronze: Flights
 - read only batch of landed flight files and writes then into bronze as a managed delta table, with an explicit schema enforced on read
 - source: aeropulse_landing_lh -> Files/flight/batch_id/*.csv
 - target: aeropulse_bronze_lh -> bronze_flight
 - load type: incremental - replace and partition by batch_id, so re-running a batch replaces only that batch's rows
 - schema handling: all columns read as strings against explicit schema, with FAILFAST on any non-comforming row-type casting and cleanup are deferred to silver
 - parameters: batch_id
 - dependencies: bronze-environment, bronze-helpers

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 3, Finished, Available, Finished, False)

## create an ingestion batch_id

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 4, Finished, Available, Finished, False)

## call environment variable

In [3]:
%run bronze-environment

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 5, Finished, Available, Finished, True)

In [4]:
source_path = f"{landing_lakehouse_path}/flight/{batch_id}/*.csv"
print(source_path)


StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 6, Finished, Available, Finished, False)

abfss://aeropulse_dev@onelake.dfs.fabric.microsoft.com/aeropulse_landing_lh.Lakehouse/Files/flight/2018_02/*.csv


## call the write_to_bronze function

In [5]:
%run bronze-helper

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 8, Finished, Available, Finished, True)

## create the flight_df schema

In [6]:
flight_schema = StructType([
    StructField('FL_DATE', StringType()), 
    StructField('OP_UNIQUE_CARRIER', StringType()),
    StructField('TAIL_NUM', StringType()), 
    StructField('OP_CARRIER_FL_NUM', StringType()),
    StructField('ORIGIN', StringType()), 
    StructField('ORIGIN_CITY_NAME', StringType()),
    StructField('DEST', StringType()), 
    StructField('DEST_CITY_NAME', StringType()),
    StructField('DEP_TIME', StringType()), 
    StructField('DEP_DELAY', StringType()), 
    StructField('TAXI_OUT', StringType()), 
    StructField('WHEELS_OFF', StringType()), 
    StructField('WHEELS_ON', StringType()),
    StructField('TAXI_IN', StringType()), 
    StructField('ARR_TIME', StringType()), 
    StructField('ARR_DELAY', StringType()), 
    StructField('CANCELLED', StringType()), 
    StructField('CANCELLATION_CODE', StringType()), 
    StructField('DIVERTED', StringType()), 
    StructField('AIR_TIME', StringType()), 
    StructField('DISTANCE', StringType()), 
    StructField('CARRIER_DELAY', StringType()), 
    StructField('WEATHER_DELAY', StringType()), 
    StructField('NAS_DELAY', StringType()), 
    StructField('SECURITY_DELAY', StringType()), 
    StructField('LATE_AIRCRAFT_DELAY', StringType()), 
    StructField('batch_id', StringType()), 
    StructField('ingested_timestamp', StringType()), 
    StructField('source_path', StringType())
    ]
)

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 9, Finished, Available, Finished, False)

## read landing lakehouse flight data

In [7]:
flight_df = spark.read.format('csv').option("header", "true").schema(flight_schema).option("mode", 'FAILFAST').load(source_path)

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 10, Finished, Available, Finished, False)

## write to bronze lakehouse

In [8]:
write_to_bronze(
    flight_df,
    target_table = "bronze_flight",
    batch_id=batch_id,
    load_type ="incremental"
)

StatementMeta(, ad500066-d7a5-489b-af43-a9caf0850519, 11, Finished, Available, Finished, False)